## Knob reference — every tunable knob

**Core model** (per-rack power, watts):
`power(r, t) = scaleᵣ · ( floor + Σⱼ pⱼ · profileⱼ(t) · 𝟙[r ∈ Sⱼ] · 𝟙[t ∈ activeⱼ] ) + noise`

**Key relations** (M/G/∞ occupancy; these explain the formulas below):
- occupancy (mean # active jobs on a rack) = `λ · E[frac] · E[d]`
- `E[power(r)] = scaleᵣ · ( floor + occupancy · E[pⱼ] )`  → this is what sets per_rack_mean & total_mean
- lognormal mean `E[d] = exp(μ + σ²/2)`  ;  beta mean `E[frac] = a/(a+b)`

| Knob (cfg field) | Represents | Formula / how set | Impacts which validation stats | Bucket |
|---|---|---|---|---|
| `idle_floor_W` | per-rack idle baseline power | from spec ≈253 kW; added to every cell before jobs | marginal **idle mode** (~0.25 MW); additive floor in per_rack_mean & total_mean | A |
| `per_rack_scale[r]` | per-rack hardware scale (homogeneity) | `scaleᵣ = meanᵣ / grand_mean` (~1; rack14=0.652); applied `power[:,r] *= scaleᵣ` | **per_rack_mean** shape across racks (relative levels) | A |
| `job_power` (`mean`,`std`) | power one job adds per rack (shared by the job's racks) | `E[pⱼ] = busy − floor ≈ 756 kW`; `pⱼ ∼ Normal(mean, std)` | marginal **busy mode** height (~1.0 MW); per_rack_mean/total_mean via `occ·E[pⱼ]`; `std` = busy-mode width | A |
| `arrival_rate_per_s` (λ) | Poisson job arrival rate (jobs/s) | `N ∼ Poisson( λ·(T+burn) )`; seeded `λ = occupancy/(E[frac]·E[d])`, `occupancy = (grand_mean−floor)/E[pⱼ]` | **level**: per_rack_mean & total_mean; busy-fraction; peak stacking; ramp density | A |
| `duration` (`μ`,`σ`) | job length, seconds (persistence) | `dⱼ ∼ Lognormal(μ, σ)`; `μ` set so `E[d] = τ·dt = 2610 s` | `E[d]` → **autocorr τ** & occupancy(level); **`σ` (tail)** → per_rack_mean sampling variance + ramp **kurtosis** | A (E[d]) / B (σ) |
| `job_size` (`a`,`b`) | job size as a **fraction of the machine** → k racks | `frac ∼ Beta(a,b)`, `E[frac] = a/(a+b)`; `k = max(1, round(frac·n_racks))` | **offdiag_corr & pc1_var_share** (synchronization via co-placement); also occupancy via `E[frac]` (level) | C |
| `placement` | which k racks a job lands on | `"scattered"` = random subset; `"contiguous"` = consecutive block (wrap) | residual **off-diagonal block structure** (real day = none → scattered); minor on aggregates | C |
| `noise_amp_W` | std of additive per-rack noise per step | `power += Normal(0, noise_amp_W)`, then clip ≥ 0 | **up** ramp_abs_mean, **down** kurtosis; **but down** corr/pc1 (rack-independent variance) → tradeoff | A |
| `burstiness` | arrival clustering beyond Poisson (reserved) | no-op at `0.0` (pure Poisson); future Hawkes / neg-binomial | (future) ramp & job-boundary burstiness; none while 0 | B |
| `_job_profile` *(generator.py, NOT a cfg field)* | intra-job power shape over active steps | flat `np.ones(n_active)`; editable to ramp-up/down or correlated wiggle | **up** ramp_abs_mean, **down** kurtosis **without** killing corr (variation shared across the job's racks → stays synchronized) | code |

**Notes**
- **noise vs `_job_profile`:** raw `noise_amp_W` fixes ramps/kurtosis but erodes corr; a shaped `_job_profile` adds within-job ramps that move *together* across a job's racks → fixes ramps **without** lowering corr. Prefer the profile.
- **Coupling:** `arrival_rate` and `job_size` BOTH move occupancy (`λ·E[frac]·E[d]`). If you change `job_size`'s mean, re-derive `arrival_rate` to keep the level, or the mean drifts.
- **per_rack_mean residual** (after the occupancy fix) is sampling noise from few/heavy-tailed jobs → lower `duration.σ`; always judge on the **multi-seed** pass-rate, never seed 0.
- **Not knobs:** `n_racks`, `n_steps`, `dt`, `seed` are `generate()` args (output shape + RNG), deliberately kept out of the config.

---

## Mathematical foundations

The synthesizer is a **marked point process / shot-noise** model. The *mechanism* is chosen as the minimal generator that reproduces all of the real day's statistics at once (bimodal marginal, persistence, rank-1 cross-rack correlation, heavy-tailed ramps); its *parameters* are then calibrated by simulation-based inference (MSM/ABC), because the likelihood is intractable (latent jobs, combinatorial marginalization — easy to **sample**, impossible to **score**). Each term rests on a named result:

| Building block | Foundation | Equation | What it provides |
|---|---|---|---|
| jobs arrive in time | **Poisson process** | `N([a,b]) ∼ Poisson(λ·(b−a))`; inter-arrivals `∼ Exp(λ)` | the arrival times `tⱼ` |
| each job carries marks | **marked point process** | `tⱼ ↦ (Sⱼ, pⱼ, dⱼ)` | rack-set, per-rack power, duration |
| power = sum of overlapping jobs | **superposition / shot noise (filtered Poisson)** | `X(t) = Σⱼ pⱼ · profileⱼ(t − tⱼ)` | the additive `Σⱼ` core |
| moments of that sum | **Campbell's theorem** | `E[X] = λ ∫ h`, `Var[X] = λ ∫ h²` (closed form) | makes MSM moment-matching tractable despite no density |
| # simultaneously-active jobs | **M/G/∞ queue occupancy** (Little's law) | per-rack `occupancy = λ·E[frac]·E[d]`, `∼ Poisson` | the mean-power **level**; insensitive to duration *shape* |
| jumps at job edges | **compound Poisson (Lévy) process** | `J(t) = Σ_{tⱼ ≤ t} pⱼ` | heavy-tailed ramps / kurtosis |
| marginals ⊥ dependence | **Sklar's theorem** | `F(x₁..x₂₅) = C(F₁,..,F₂₅)` | justifies hardware (`scaleᵣ`) and scheduling (`Sⱼ`) as *separable* factors (used as a soundness check, NOT a fitted copula) |
| residual scatter | **i.i.d. Gaussian noise** | `+ N(0, noise_amp_W)` | the rank-defying / un-shared component |

Construction is two-sided: **bottom-up** from the HPC physical/scheduling story (a job occupies a rack-set for a duration, drawing power; a rack's power = idle floor + active jobs on it) and **top-down** constrained to match the frozen `spec/regime_A.json` statistics. Note the synthesizer produces the *exogenous input* (compute power per CDU group); it is the demand driving the FMU/controller, **not** a target any downstream network is trained to predict.

### Occupancy and the level calibration — why `occupancy = λ·E[frac]·E[d]`, and how `λ` is set

**Per-rack arrival rate (Poisson thinning).** Jobs arrive globally as a Poisson process of rate `λ`. A given job touches rack `r` only if `r ∈ Sⱼ`; for scattered placement `P(r ∈ Sⱼ) = kⱼ/n_racks = fracⱼ`, so averaged over jobs the probability is `E[frac]`. The **thinning theorem** (keep each point of a Poisson process independently with probability `p` → a Poisson process of rate `λ·p`) gives the rate of jobs landing on one rack:
```
λ_r = λ · E[frac]
```
Intuition: of the `λ` jobs/s arriving, only the fraction `E[frac]` touch any given rack. Units: `[jobs/s] · [prob] = [jobs/s on this rack]`.

**Occupancy (Little's law).** `L = λ_r · W` — arrival rate into the rack × mean time each stays. Under M/G/∞ there is no queueing, so time-in-system = duration, `W = E[d]`. Hence
```
occupancy = λ · E[frac] · E[d]      (dimensionless: [1/s]·[—]·[s])
```
The **mean** uses only `E[d]`, not the duration *shape* (the M/G/∞ insensitivity property) — which is why `E[d]` is bucket A (pinned by `τ`) while the tail `σ` is a separate flat direction.

**Level.** Each active job adds `E[pⱼ]` on average, so `E[power(r)] = scaleᵣ · ( floor + occupancy · E[pⱼ] )`.

**How `λ` is determined — solved *last*, from the mean.** Dependency order (each step needs the one above it):
1. `E[d] ← τ·dt = 174·15 = 2610 s`  (autocorrelation; bucket A)
2. `E[frac] = a/(a+b) ← job_size` Beta, forward-calibrated to the corr/pc1 target; regime A ≈ 0.96 (near full-machine)  (bucket C)
3. `occupancy ← (grand_mean − floor)/E[pⱼ] = (645 − 253)/756 ≈ 0.519`  (invert the level to the spec mean)
4. `λ = occupancy / (E[frac]·E[d]) = 0.519 / (0.964 · 2610) ≈ 2.06e-4 /s  (≈ 18 jobs/day)`

So `λ` is whatever arrival rate makes the mean power land on `grand_mean`, *given* the size and duration already chosen. (Patched derivation: the earlier **busy-fraction** constraint set `occupancy = ln2 ≈ 0.69` and overshot the mean by ~18%; the two constraints disagreed and the mean one won.) **Coupling:** change `E[frac]` or `E[d]` and `λ` must be re-solved, or the level drifts.

---

## Known limitation — Poisson arrivals / M/G/∞ (future exploration)

Regime A models arrivals as a **homogeneous Poisson process** and occupancy via the **M/G/∞** queue (infinite servers ⇒ unlimited concurrent jobs). Real HPC submission violates all three assumptions:
- **non-stationary rate** — diurnal/weekly human cycles (`λ(t)`, not constant `λ`);
- **clustering** — job arrays / batch submits / dependency chains ⇒ self-exciting (Hawkes) arrivals, *super*-Poisson occupancy;
- **occupancy-dependent regulation** — a finite machine queues/blocks jobs near capacity (SLURM) and users self-throttle ⇒ negative feedback `λ(occupancy)`, *sub*-Poisson (anti-bunched) occupancy. The true system is `M/G/c/c` (Erlang-loss) or queued `M/G/k`, **not** `M/G/∞`; the `∞` approximation is only good when occupancy ≪ capacity.

**Why Poisson is still the regime-A baseline:** (1) max-entropy — given only the rate, it assumes the least; (2) the arrival law is a **flat / under-identified direction** of the regime-A objective — one calm day's aggregate stats cannot distinguish Poisson from mild alternatives; (3) we calibrate to aggregate summary stats, not to the arrival process; (4) this day is unsaturated (busy-fraction ≈ 0.4–0.5), where the `∞`-server approximation is least wrong. M/G/∞ is used here mainly as a closed-form **calibration device** (it yields `occupancy = λ·E[frac]·E[d]`), not as a claim about the scheduler.

**Connection to a known discrepancy:** Poisson occupancy cannot simultaneously give a high busy-fraction AND no peak-stacking — but real traces run *more regularly* (sub-Poisson). Occupancy-dependent regulation is the natural **generative explanation** for that sharp, no-stacking busy level.

**Future improvement (only if the busy-fraction / stacking / occupancy-variance stats won't pass within tolerance):** swap the arrival mechanism for one of — **capped concurrency** (`M/G/c/c` loss), a **renewal process with sub-exponential gaps** (anti-bunching), or an **occupancy-throttled rate** `λ(occupancy)`. These reduce occupancy variance below Poisson. (The `burstiness` knob is the *opposite* axis — Hawkes super-Poisson clustering — reserved for regime B; real HPC exhibits both and the net dispersion is empirical.) Do not build preemptively: it is a flat direction until a specific stat forces it.